# 반입 엔진 — 남의 자료를 들이는 기계를 돌려 본다

PR #32 에서 **규격**(무엇이 맞는 모양인가)을 세웠고, 여기서는 그 규격을 실제로 집행하는
**엔진**을 돌립니다. 정제기 20종을 구현하고, 시점을 파생하고, 판정해서 DB 에 담습니다.

이 노트북이 답하는 것

1. 지저분한 파일이 들어오면 무엇을 어떻게 고치는가 — **그리고 그걸 어떻게 기록하는가**
2. 미래참조를 심어 보내면 잡히는가
3. 🔴 **우리 자료를 우리 검사기에 넣으면 통과하는가** — 이번에 통과하지 못했고, 그 이유를 찾았습니다

> **읽는 법**: 위에서부터 그냥 실행하면 됩니다. DB 를 읽기만 하고 쓰지 않습니다
> (마지막 적재 절만 임시 DB 를 씁니다).

In [1]:
import sqlite3
import sys
import tempfile
import time
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():      # 노트북을 어디서 열든 루트를 찾는다
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# 경로를 세운 뒤에 import 한다 — 노트북을 어디서 열든 저장소 모듈을 찾게 하려면 순서가 이래야 한다
from common.paths import krx_db_path  # noqa: E402
from ingest.inbox import cleaners, store  # noqa: E402
from ingest.inbox.engine import guess_kind, inspect_file, read_table  # noqa: E402
from ingest.inbox.report import render_markdown  # noqa: E402

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
print("루트    :", ROOT.name)
print("정제기  :", len(cleaners.CLEANERS), "종")
print("pandas  :", pd.__version__)

루트    : Alpha_Stack
정제기  : 20 종
pandas  : 3.0.5


---

## 1. 정제기 20종 — 무엇을 무엇으로 바꾸는가

규격에는 `["strip", "strip_comma", "to_int"]` 처럼 **이름만** 적혀 있었습니다. 이번에 그 20개를
실제로 구현했습니다.

먼저 하나씩 **따로** 걸어 각자 무엇만 건드리는지 봅니다 (사슬로 잇는 것은 다음 절입니다).
같은 값 여섯 개에 정제기를 하나씩 대면, 자기 일이 아닌 값은 그대로 두는 것이 보입니다.

In [2]:
표본 = pd.Series(["  81,000 ", "(1,234)", "2.47%", "5930", "005930.KS", "일이삼"])
print(f"{'(원본)':20s} {[str(v) for v in 표본.tolist()]}")
print()
for 이름 in ["strip", "strip_comma", "paren_to_negative", "strip_percent",
           "zfill6", "drop_suffix_ks_kq"]:
    결과 = cleaners.CLEANERS[이름](표본)
    print(f"{이름:20s} {[str(v) for v in 결과.values.tolist()]}")

(원본)                 ['  81,000 ', '(1,234)', '2.47%', '5930', '005930.KS', '일이삼']

strip                ['81,000', '(1,234)', '2.47%', '5930', '005930.KS', '일이삼']
strip_comma          ['  81000 ', '(1234)', '2.47%', '5930', '005930.KS', '일이삼']
paren_to_negative    ['  81,000 ', '-1,234', '2.47%', '5930', '005930.KS', '일이삼']
strip_percent        ['81,000', '(1,234)', '2.47', '5930', '005930.KS', '일이삼']
zfill6               ['81,000', '(1,234)', '2.47%', '005930', '005930.KS', '일이삼']
drop_suffix_ks_kq    ['81,000', '(1,234)', '2.47%', '5930', '005930', '일이삼']


### 순서가 뜻을 갖는다

`strip_comma` 를 `to_int` 보다 **먼저** 걸어야 `"1,234"` 를 숫자로 읽습니다. 규격이 순서를
적어 둔 이유입니다.

In [3]:
값 = pd.Series(["1,234", "(5,678)", "일이삼", None])
값 = cleaners.normalize_missing(값, ["", "-", "NA"])

바른순서, 기록, 실패, 바뀜 = cleaners.apply_chain(
    값, ["strip", "strip_comma", "paren_to_negative", "to_int"], column="금액")
print("바른 순서 :",바른순서.tolist())

거꾸로, _, _, _ = cleaners.apply_chain(값, ["to_int", "strip_comma"], column="금액")
print("뒤집으면  :", 거꾸로.tolist(), "  ← 쉼표를 못 뗀 채 숫자로 읽어 전부 실패")

바른 순서 : [1234, -5678, <NA>, <NA>]
뒤집으면  : [None, None, None, None]   ← 쉼표를 못 뗀 채 숫자로 읽어 전부 실패


### 기록 — 전량 집계 + 표본 20건

**무엇을 무엇으로 바꿨는지 남기지 않는 정제는 정제가 아니라 훼손입니다.** 나중에 값이 이상할 때
*출처가 그랬는지 우리가 그랬는지* 알 수 없으니까요.

상세도는 [Great Expectations](https://greatexpectations.io) 의 기본 결과 형식(`SUMMARY`)을
따랐습니다 — 집계는 전량, 실제 값 목록은 20건까지. 팀원이 몇십만 행을 주면 전량 기록이 원본보다
커지고, 건수만 남기면 *"318건 변경"* 이 `zfill6` 때문인지 엉뚱한 절단 때문인지 알 수 없습니다.

In [4]:
for 항목 in 기록.to_list():
    print(f"{항목['cleaner']:20s} 바뀜 {항목['changed']:>3} · 못읽음 {항목['failed']:>3} "
          f"· 표본 {항목['samples']} · 실패값 {항목['failed_samples']}")
print()
print("실패한 자리:", 실패.tolist(), " ← 값이 있었는데 못 읽은 것. 빈 칸과 구별한다")

strip_comma          바뀜   2 · 못읽음   0 · 표본 [{'from': '1,234', 'to': '1234'}, {'from': '(5,678)', 'to': '(5678)'}] · 실패값 []
paren_to_negative    바뀜   1 · 못읽음   0 · 표본 [{'from': '(5678)', 'to': '-5678'}] · 실패값 []
to_int               바뀜   0 · 못읽음   1 · 표본 [] · 실패값 ['일이삼']

실패한 자리: [False, False, True, False]  ← 값이 있었는데 못 읽은 것. 빈 칸과 구별한다


---

## 2. 지저분한 파일 하나를 통째로 들여 본다

팀원이 실제로 줄 법한 모양입니다 — 한글 칸 이름, 천단위 쉼표, 엑셀이 지운 앞자리 0,
거래정지 종목의 `-`, 그리고 **일부러 심은 오류 세 개**.

In [5]:
지저분한파일 = Path(tempfile.gettempdir()) / "팀원_시세.csv"
지저분한파일.write_text('''날짜,종목코드,종목명,시장,시가,고가,저가,종가,등락률,거래량,메모
2021-01-04,5930,삼성전자,코스피,"81,000","84,400","80,200","83,000",2.47%,"38,655,276",첫날
2021-01-05,005930,삼성전자,유가증권,"81,600","83,900","81,600","83,900",1.08%,"35,335,669",
2021-01-06,035720,카카오,코스닥,"-","-","-","39,000",0.0%,"1,234,567",거래정지
2021-01-06,373220,LG에너지,코스피,"400,000","390,000","395,000","398,000",1.0%,"500,000",고가<시가
2021-01-07,999999,없는종목,NASDAQ,"1,000","1,100","900","1,050",5.0%,"100",시장이상
2021-01-08,005930,삼성전자,코스피,"82,000","83,000","81,000",일이삼,0.5%,"1,000",종가못읽음
''', encoding="utf-8")

판정 = inspect_file(지저분한파일, kind="ohlcv_stock")
print(f"전체 {판정.rows_total}행 → 합격 {len(판정.accepted)} · 격리 {len(판정.quarantined)}")

전체 6행 → 합격 3 · 격리 3


In [6]:
print("칸 이름 매핑:")
for 원본, 규격 in 판정.report["columns"]["mapped"].items():
    print(f"  {원본:6s} → {규격}")
print()
print("규격 밖 칸(버리지 않고 extras 로 보존):", 판정.report["columns"]["extras"])

칸 이름 매핑:
  날짜     → bas_dd
  종목코드   → code
  종목명    → name
  시장     → market
  시가     → open
  고가     → high
  저가     → low
  종가     → close
  등락률    → change_rate
  거래량    → volume

규격 밖 칸(버리지 않고 extras 로 보존): ['메모']


In [7]:
pd.DataFrame([r["payload"] for r in 판정.accepted.to_dict("records")])[
    ["bas_dd", "code", "name", "market", "open", "high", "low", "close", "volume"]]

,bas_dd,code,name,market,open,high,low,close,volume
0,20210104,005930,삼성전자,KOSPI,81000.0,84400.0,80200.0,83000,38655276
1,20210105,005930,삼성전자,KOSPI,81600.0,83900.0,81600.0,83900,35335669
2,20210106,035720,카카오,KOSDAQ,NaN,NaN,NaN,39000,1234567


**거래정지 종목(카카오, 시고저가가 `-`)이 합격했다**는 점이 중요합니다. 결측 표기를 오류로
세면 정상 자료가 통째로 사라집니다. 규격의 `missingValues` 를 정제보다 **먼저** 적용하는
이유입니다.

In [8]:
for 행 in 판정.quarantined.to_dict("records"):
    사유 = " · ".join(v["rule"] for v in 행["violations"])
    print(f"행 {행['row_no']}  {행['raw']['종목명']:8s}  →  {사유}")
    print(f"        메모: {행['raw']['메모']}")

행 4  LG에너지     →  high_ge_low · high_ge_open_close
        메모: 고가<시가
행 5  없는종목      →  market.enum
        메모: 시장이상
행 6  삼성전자      →  cleaner.failed · close.required
        메모: 종가못읽음


심어 둔 오류 세 개가 전부 잡혔습니다. 격리된 행은 **원본까지 함께** 담깁니다 — 사람이 고쳐
다시 넣으려면 우리가 정제하기 전 값이 필요하니까요.

---

## 3. 미래참조를 심어 보낸다

여기가 이 엔진에서 가장 중요한 자리입니다. 값이 틀린 파일은 규칙에 걸리지만, **시점이 틀린
파일은 아무 데도 걸리지 않습니다** — 형식도 맞고 규칙도 통과하고 성능만 좋아집니다.

그래서 팀원이 채워 온 `eff_dd`(뉴스가 처음 쓰일 수 있는 거래일)를 **그대로 믿지 않고**
우리가 같은 계산을 해서 대조합니다.

In [9]:
뉴스파일 = Path(tempfile.gettempdir()) / "팀원_뉴스.csv"
뉴스파일.write_text('''발행시각,제목,요약,링크,언론사,검색어,유효거래일
2021-01-04T08:15:00+09:00,장 시작 전 기사,요약1,https://n.news/1?utm_source=naver,한국경제,반도체,
2021-01-04T16:00:00+09:00,장 마감 후 기사,요약2,https://n.news/2,매일경제,반도체,20210104
2021-01-02T10:00:00+09:00,주말 기사,요약3,https://n.news/3,연합뉴스,반도체,
"Mon, 04 Jan 2021 09:30:00 +0900",장중 기사,<b>삼성</b> 실적,https://n.news/4?id=99&utm_medium=x,조선비즈,반도체,
''', encoding="utf-8")

뉴스판정 = inspect_file(뉴스파일, kind="news")
for 행 in 뉴스판정.accepted.to_dict("records"):
    p = 행["payload"]
    print(f"{p['pub_dt']}  →  eff_dd {p['eff_dd']}   {p['title']}")
print()
for 행 in 뉴스판정.quarantined.to_dict("records"):
    for v in 행["violations"]:
        print(f"격리 {행['row_no']}행 [{v['rule']}] {v['note'][:70]}")

2021-01-04T08:15:00+09:00  →  eff_dd 20210104   장 시작 전 기사
2021-01-02T10:00:00+09:00  →  eff_dd 20210104   주말 기사
2021-01-04T09:30:00+09:00  →  eff_dd 20210105   장중 기사

격리 2행 [lookahead] eff_dd 가 규칙보다 이르다 (적힌 값 20210104 · 규칙 20210105) — 미래참조
격리 2행 [eff_dd_same_day_needs_premarket] 🔴 **이 규격에서 가장 중요한 규칙이다.** 유효 거래일이 발행일과 같은 날이면 발행 시각이 그날 08:30(시가 단일가 호


배정 규칙은 규격이 적어 둔 그대로입니다.

| 발행 시각 | 유효 거래일 | 왜 |
|---|---|---|
| 08:30 **미만** | 그날 | 시가 단일가 호가가 08:30 부터 쌓인다 |
| 08:30 **이상** | 다음 거래일 | 장중·장마감 후 기사가 여기로 모인다 |
| 비거래일 | 이후 첫 거래일 | 주말·공휴일 |
| 정확히 `00:00:00` | 다음 거래일 | 날짜만 있는 자료 — 모르는 쪽을 늦게 잡는다 |

두 번째 행은 장 마감 후 기사에 **발행일을 그대로** 적어 왔습니다. 그러면 그날 시가에 쓰게
되는데, 그건 정확히 미래참조입니다. 검사가 **두 겹으로** 잡았습니다 — 우리 파생 대조와
규격 자체 규칙.

거래일 판정은 계산이 아니라 **실측**입니다. `daily_price.bas_dd` 에 있는 날이 거래일입니다.

In [10]:
import datetime as dt

from common.trading_calendar import load_session_days, next_session, session_span

달력 = load_session_days()
첫날, 끝날 = session_span()
print(f"거래일 달력: {len(달력):,}일 ({첫날}~{끝날}) — 계산이 아니라 실측 기록")
print()
개발구간 = {d for d in 달력 if d <= "20210831"}
평일 = sum(1 for n in range((dt.date(2021, 8, 31) - dt.date(2010, 1, 4)).days + 1)
         if (dt.date(2010, 1, 4) + dt.timedelta(days=n)).weekday() < 5)
print(f"개발구간 평일 {평일:,}일 · 실제 거래일 {len(개발구간):,}일 "
      f"→ 공휴일 {평일 - len(개발구간):,}일 ({(평일-len(개발구간))/평일*100:.1f}%)")
print()
print("2021-01-01 은 금요일이지만 신정이라 휴장:")
print("  next_session('20210101') =", next_session("20210101"))

거래일 달력: 4,097일 (20100104~20260825) — 계산이 아니라 실측 기록

개발구간 평일 3,042일 · 실제 거래일 2,880일 → 공휴일 162일 (5.3%)

2021-01-01 은 금요일이지만 신정이라 휴장:
  next_session('20210101') = 20210104


🔴 주말만 거르는 달력을 쓰면 **162일이 어긋납니다.** 그 162일은 명절·공휴일이고, 하필
실적 발표와 뉴스가 몰리는 연휴 전후입니다.

### 거시 지표 — 지연이 출처마다 다르다

`known_from`(그 통계를 처음 알 수 있었던 날)은 발표일이 있으면 그것을, 없으면
`참조기간 시작 + 지연` 으로 채웁니다. **지연표가 출처마다 다릅니다.**

In [11]:
거시파일 = Path(tempfile.gettempdir()) / "팀원_거시.csv"
거시파일.write_text('''출처,계열코드,계열명,주기,기간,기간시작,기간종료,값,단위,발표일,사용가능일
fred,CPIAUCSL,미국 소비자물가,월,2021-07,20210701,20210731,273.003,Index,,
한국은행,722Y001,기준금리,monthly,2021-07,20210701,20210731,0.5,%,,
FRED,GDPC1,미국 실질GDP,분기,2021Q2,20210401,20210630,19368.3,Bil,,20210702
통계청,DT_1J22042,소비자물가지수,월,2021-07,20210701,20210731,107.61,Index,20210803,
''', encoding="utf-8")

거시판정 = inspect_file(거시파일, kind="macro")
for 행 in 거시판정.accepted.to_dict("records"):
    p = 행["payload"]
    print(f"{p['source']:6s} {p['freq']}  기간시작 {p['period_start']}"
          f"  →  known_from {p['known_from']} ({p['known_from_basis']})")
print()
for 행 in 거시판정.quarantined.to_dict("records"):
    for v in 행["violations"]:
        print(f"격리: {v['note'][:90]}")

FRED   M  기간시작 20210701  →  known_from 20210815 (estimate)
ECOS   M  기간시작 20210701  →  known_from 20210802 (estimate)
KOSIS  M  기간시작 20210701  →  known_from 20210803 (release)

격리: known_from 이 규칙보다 이르다 (적힌 값 20210702 · 규칙 20210809) — 미래참조


같은 월간 지표인데 **ECOS 는 07-01+32일 = 08-02, FRED 는 07-01+45일 = 08-15** 입니다.
미 CPI 는 익월 11~13일에 나오는데 한국 관행(익월 2일)에 맞춘 32일을 쓰면 **9~11일을 미리
보게 됩니다.**

세 번째 행은 FRED 분기 지표에 한국 관행으로 이른 날짜를 적어 왔고, 격리됐습니다.

> ⚠️ FRED 계열별 실제 공표 일정은 **확인하지 못했습니다.** 월 45일·분기 130일은 관행에
> 여유를 얹은 추정입니다. 계열마다 다르므로 45일보다 늦게 나오는 월간 계열이 있으면 그만큼 샙니다.

---

## 4. 🔴 우리 자료를 우리 검사기에 넣으면 통과하는가

여기가 이 노트북의 핵심입니다. 팀원 파일로만 시험하면 **검사기가 전부 격리해도 "안전하게"
보입니다.** 우리가 이미 갖고 있는 자료를 거부한다면, 팀원 자료도 같은 이유로 거부합니다.

`daily_price` 에서 무작위로 뽑아 **팀원이 줄 법한 모양으로 내보낸 뒤** 다시 들여 봅니다.

In [12]:
conn = sqlite3.connect(krx_db_path())
원본 = pd.read_sql_query(
    """SELECT bas_dd, code, name, market, open, high, low, close,
              change_rate, volume, value, market_cap
       FROM daily_price WHERE bas_dd <= '20210831'
       ORDER BY random() LIMIT 40000""", conn)
conn.close()
print(f"뽑은 행 {len(원본):,} · {원본['bas_dd'].min()}~{원본['bas_dd'].max()}")

def 쉼표(v):
    return f"{v:,}" if pd.notna(v) else ""

내보낸것 = pd.DataFrame({
    "날짜": 원본["bas_dd"], "종목코드": 원본["code"], "종목명": 원본["name"],
    "시장": 원본["market"],
    "시가": 원본["open"].map(쉼표), "고가": 원본["high"].map(쉼표),
    "저가": 원본["low"].map(쉼표), "종가": 원본["close"].map(쉼표),
    "등락률": 원본["change_rate"].map(lambda v: f"{v}%" if pd.notna(v) else ""),
    "거래량": 원본["volume"].map(쉼표), "거래대금": 원본["value"].map(쉼표),
    "시가총액": 원본["market_cap"].map(쉼표),
})
왕복파일 = Path(tempfile.gettempdir()) / "왕복.csv"
내보낸것.to_csv(왕복파일, index=False, encoding="utf-8")

시작 = time.perf_counter()
왕복 = inspect_file(왕복파일, kind="ohlcv_stock")
걸린시간 = time.perf_counter() - 시작
print(f"검사 {걸린시간:.1f}초 ({왕복.rows_total/걸린시간:,.0f}행/초)")
print(f"합격 {len(왕복.accepted):,} ({len(왕복.accepted)/왕복.rows_total*100:.3f}%) "
      f"· 격리 {len(왕복.quarantined):,}")
print("격리 사유:", 왕복.report["quarantine_reasons"] or "없음")

뽑은 행 40,000 · 20100104~20210831
검사 15.8초 (2,530행/초)
합격 40,000 (100.000%) · 격리 0
격리 사유: 없음


### 처음 돌렸을 때는 통과하지 못했습니다

규격 v1.1 로 같은 시험을 했을 때 **`code.pattern` 위반 421행(0.351%)** 이 나왔습니다.
값을 열어 보니 전부 실재하는 정상 종목이었습니다.

In [13]:
conn = sqlite3.connect(krx_db_path())
영문코드 = pd.read_sql_query(
    """SELECT code, min(name) name, count(*) n, min(bas_dd) 처음, max(bas_dd) 마지막
       FROM daily_price WHERE code GLOB '*[^0-9]*'
       GROUP BY code ORDER BY n DESC LIMIT 10""", conn)
전체 = conn.execute("SELECT count(*) FROM daily_price").fetchone()[0]
걸린행 = conn.execute("SELECT count(*) FROM daily_price WHERE code GLOB '*[^0-9]*'").fetchone()[0]
걸린종 = conn.execute("SELECT count(DISTINCT code) FROM daily_price "
                    "WHERE code GLOB '*[^0-9]*'").fetchone()[0]
자리별 = {n: conn.execute(f"SELECT count(DISTINCT code) FROM daily_price "
                       f"WHERE substr(code,{n},1) GLOB '[^0-9]'").fetchone()[0]
        for n in range(1, 7)}
conn.close()

print(f"`^[0-9]{{6}}$` 가 막던 것: {걸린행:,}행 / {전체:,} ({걸린행/전체*100:.2f}%) · {걸린종}종")
print("자리별 영문 출현:", {f"{k}번째": v for k, v in 자리별.items()})
영문코드

`^[0-9]{6}$` 가 막던 것: 56,190행 / 9,209,812 (0.61%) · 84종
자리별 영문 출현: {'1번째': 0, '2번째': 0, '3번째': 0, '4번째': 0, '5번째': 57, '6번째': 28}


,code,name,n,처음,마지막
0,00781K,코리아써키트2우B,3237,20130617,20260825
1,18064K,한진칼우,3173,20130916,20260825
2,03473K,SK우,2704,20150817,20260825
3,02826K,삼성물산우B,2683,20150915,20260825
4,00088K,한화3우B,2416,20161019,20260825
5,26490K,크라운제과우,2296,20170411,20260825
6,00499K,롯데지주우,2164,20171030,20260825
7,28513K,SK케미칼우,2118,20180105,20260825
8,00680K,미래에셋대우2우B,2073,20180314,20260825
9,00806K,대덕1우,1884,20181219,20260825


**KRX 가 여섯 자리를 다 써서 영문을 섞기 시작한 것**입니다.

- **신형우선주**는 끝자리가 영문입니다 — `00781K`(코리아써키트2우B) · `08537M`(루트로닉3우C)
- **최근 상장·스팩**은 다섯째 자리가 영문입니다 — `0004Y0`(디비금융제14호스팩) · `0007C0`(아크릴)
- 1~4번째 자리는 3,677종 **전부 숫자**였습니다

그래서 v1.2 에서 `^[0-9]{6}$` → **`^[0-9]{4}[0-9A-Z]{2}$`** 로 넓혔습니다.

**왜 `^[0-9A-Z]{6}$` 로 다 열지 않았나.** `pykrx`·`FinanceDataReader` 같은 라이브러리는
종목코드에 정규식을 **아예 걸지 않습니다** — KRX 가 주는 문자열을 그대로 씁니다. 그 관행대로면
패턴을 지우는 게 맞지만, 반입은 남의 파일을 받는 자리라 오타는 잡아야 합니다. 그래서 관행만큼
느슨하되 **실측이 보장하는 만큼만** 조였습니다.

In [14]:
conn = sqlite3.connect(krx_db_path())
우선주 = pd.read_sql_query(
    """SELECT bas_dd, code, name, market, open, high, low, close, change_rate, volume
       FROM daily_price WHERE code GLOB '*[^0-9]*' AND bas_dd <= '20210831'""", conn)
conn.close()

우선주파일 = Path(tempfile.gettempdir()) / "영문코드.csv"
우선주.rename(columns={"bas_dd": "날짜", "code": "종목코드", "name": "종목명", "market": "시장",
                    "open": "시가", "high": "고가", "low": "저가", "close": "종가",
                    "change_rate": "등락률", "volume": "거래량"}).to_csv(
    우선주파일, index=False, encoding="utf-8")

우선주판정 = inspect_file(우선주파일, kind="ohlcv_stock")
print(f"영문 코드만 모아 왕복: {우선주판정.rows_total:,}행 → "
      f"합격 {len(우선주판정.accepted):,} · 격리 {len(우선주판정.quarantined):,}")
print("v1.1 이었다면 이 전부가 격리됐습니다.")

영문 코드만 모아 왕복: 19,297행 → 합격 19,297 · 격리 0
v1.1 이었다면 이 전부가 격리됐습니다.


### 경고는 격리가 아니다

`zero_ohlc`(시고저가가 모두 0)는 우리 자료에 2.7% 있습니다. 거래정지 종목이라 **정상**이고,
이걸 `error` 로 두면 멀쩡한 자료가 통째로 격리됩니다. 규격이 `warn` 으로 둔 이유입니다.

In [15]:
경고 = [t for t in 왕복.report["row_rules"] if t["severity"] != "error" and t["violations"]]
for 항목 in 경고:
    print(f"{항목['rule']:24s} {항목['violations']:,}행 "
          f"({항목['violations']/왕복.rows_total*100:.2f}%)")
print()
못잰것 = [t for t in 왕복.report["row_rules"] if t.get("skipped")]
print("재지 못한 규칙:", [t["rule"] for t in 못잰것] or "없음 (전부 쟀다)")

zero_ohlc                1,150행 (2.88%)

재지 못한 규칙: 없음 (전부 쟀다)


---

## 5. 담기 — `inbox_batch` · `inbox_accepted` · `inbox_quarantine`

판정을 `krx_cache.db` 에 담습니다. **`daily_price` 920만 행은 건드리지 않습니다** —
그건 우리가 16년치를 받아 쌓은 것이고 반입은 남이 준 자료라, 한 표에 섞으면
*"이 값은 누가 어디서 가져왔나"* 를 되짚을 수 없습니다.

표 모양은 [Airbyte 의 raw table](https://docs.airbyte.com/using-airbyte/core-concepts/typing-deduping)
을 따랐습니다 — 메타는 칸으로, 행 자체는 JSON 으로. 규격 5장의 칸이 서로 겹치지 않아
한 표에 펴면 71칸에 대부분이 NULL 이 되고, 종류를 하나 더 받을 때마다 마이그레이션이 붙습니다.

In [16]:
from ingest.store.migrations import migrate_path  # noqa: E402

임시DB = Path(tempfile.mkdtemp()) / "시연.db"
migrate_path(임시DB)

묶음 = store.load_result(판정, 지저분한파일, db_path=임시DB, contributor="오준영")
print("묶음 번호:", 묶음)

conn = sqlite3.connect(임시DB)
conn.row_factory = sqlite3.Row
행 = conn.execute("SELECT * FROM inbox_batch").fetchone()
print(f"  {행['kind']} · 전체 {행['rows_total']} · 합격 {행['rows_accepted']} "
      f"· 격리 {행['rows_quarantined']} · 보낸 사람 {행['contributor']}")
print()
print("합격 행에 남는 것:")
for r in conn.execute("SELECT row_no, changes, extras FROM inbox_accepted ORDER BY row_no LIMIT 2"):
    print(f"  행 {r['row_no']} 손댄 칸 {r['changes']}")
    print(f"        규격 밖 칸 {r['extras']}")
conn.close()

묶음 번호: 20260901T113957-4f37fdf88e98
  ohlcv_stock · 전체 6 · 합격 3 · 격리 3 · 보낸 사람 오준영

합격 행에 남는 것:
  행 1 손댄 칸 ["bas_dd", "change_rate", "close", "code", "high", "low", "market", "open", "volume"]
        규격 밖 칸 {"메모": "첫날"}
  행 2 손댄 칸 ["bas_dd", "change_rate", "close", "high", "low", "market", "open", "volume"]
        규격 밖 칸 {"메모": null}


In [17]:
store.accepted_frame("ohlcv_stock", db_path=임시DB)[
    ["bas_dd", "code", "name", "market", "close", "volume", "_batch_id"]]

,bas_dd,code,name,market,close,volume,_batch_id
0,20210104,005930,삼성전자,KOSPI,83000,38655276,20260901T113957-4f37fdf88e98
1,20210105,005930,삼성전자,KOSPI,83900,35335669,20260901T113957-4f37fdf88e98
2,20210106,035720,카카오,KOSDAQ,39000,1234567,20260901T113957-4f37fdf88e98


같은 파일을 두 번 들이지 않습니다. 판단은 이름이 아니라 **내용 지문(SHA-256)** 입니다 —
이름은 팀원이 바꿔 올리고, 수정 시각은 내려받을 때마다 새로 찍힙니다.

In [18]:
지문 = store.file_sha256(지저분한파일)
본적있나 = store.already_ingested(지문, 임시DB)
보여줄것 = ("kind", "rows_total", "finished_at")
print("이미 들였나:", {k: v for k, v in 본적있나.items() if k in 보여줄것})

이름만바꾼사본 = Path(tempfile.gettempdir()) / "다른이름.csv"
이름만바꾼사본.write_bytes(지저분한파일.read_bytes())
print("이름만 바꾼 사본의 지문이 같은가:", store.file_sha256(이름만바꾼사본) == 지문)

이미 들였나: {'kind': 'ohlcv_stock', 'rows_total': 6, 'finished_at': '2026-09-01T11:39:57+09:00'}
이름만 바꾼 사본의 지문이 같은가: True


---

## 6. 팀원에게 줄 답 — `reports/inbox/`

판정 결과를 받을 사람은 팀원이고, 그 사람이 알고 싶은 것은 `{"rule": "high_ge_low"}` 가 아니라
*"내 파일 6행 중 3행이 안 들어갔고, 그중 하나는 고가가 저가보다 낮아서"* 입니다.

🔴 **보고서에 자료를 싣지 않습니다.** 이 저장소는 PUBLIC 이고 시세는 KRX 이용약관 제11조 ②가
제3자 제공을 금지합니다. 담는 것은 **판정과 표본 20건**까지입니다.

In [19]:
글 = render_markdown(판정, batch_id=묶음, contributor="오준영")
print(글[:2600])

# 반입 판정 — 팀원_시세.csv

- **종류**: `ohlcv_stock` (규격 v1.2)
- **보낸 사람**: 오준영
- **묶음**: `20260901T113957-4f37fdf88e98`
- **검사 시각**: 2026-09-01 11:39 KST

## 결과

⚠️ **6행 중 3행이 들어오고 3행(50.00%)이 격리됐습니다.**

| | 행 수 | 비율 |
|---|---:|---:|
| 들어옴 | 3 | 50.00% |
| 격리 | 3 | 50.00% |
| 합계 | 6 | 100% |

## 왜 격리됐나

| 사유 | 행 수 | 무슨 뜻인가 |
|---|---:|---|
| `high_ge_low` | 1 | 고가가 저가보다 낮으면 두 칸이 뒤바뀐 것이다. 값만 보고는 어느 쪽이 옳은지 알 수 없어 고치지 않고 격리한다. 우리 daily_price 는 이 칸들이 널이 아니지만(9,209,812행 전부 채워져 있다) ... |
| `high_ge_open_close` | 1 | 고가는 그날 가장 비쌌던 값이므로 시가·종가보다 작을 수 없다. ⚠️ `high == 0` 탈출구가 없으면 이 규칙 하나가 우리 자료 283,468행(3.1%)을 격리한다 — 거래정지 종목은 시고저가 0 인데 ... |
| `market.enum` | 1 | 시장구분이 규격이 정한 값이 아니다 |
| `cleaner.failed` | 1 | 값이 있었는데 규격이 정한 형으로 읽지 못했다 |
| `close.required` | 1 | 종가는 반드시 있어야 한다 |

## 우리가 손댄 것

보낸 파일을 그대로 담지 않고 규격에 맞춰 다듬었습니다. **무엇을 무엇으로 바꿨는지 전부 적습니다** — 값이 이상해 보일 때 출처가 그런 것인지 우리가 그런 것인지 알 수 있어야 하니까요.

| 칸 | 손질 | 바뀐 행 | 못 읽음 | 예 (앞 3건) |
|---|---|---:|---:|---|
| `bas_dd` | `to_yyyymmdd` | 6 | 0 | `2021-01

---

## 7. 종류를 어떻게 아는가

로컬 `data/inbox/<종류>/` 는 **폴더가 알려 줍니다.** 그런데 HuggingFace 는
`inbox/<본인이름>/` 이라 종류 폴더가 없습니다(팀원 가이드가 그렇게 안내했습니다).

파일 이름으로 찍지 않고 **규격 5장에 다 대 보고 잽니다.** 애매하면 정하지 않고 물어봅니다 —
틀린 규격으로 검사하면 멀쩡한 파일이 통째로 격리되고, 팀원은 자기 자료가 잘못됐다고 오해합니다.

In [20]:
for 이름, 파일 in [("시세", 지저분한파일), ("뉴스", 뉴스파일), ("거시", 거시파일)]:
    표 = read_table(파일)
    정해진것, 점수표 = guess_kind(표)
    낱개 = " · ".join(f"{s['kind']} {s['score']:.2f}" for s in 점수표[:3])
    print(f"{이름:4s} → {str(정해진것):14s}  {낱개}")

print()
알수없는것, 점수 = guess_kind(pd.DataFrame(columns=["뭔가", "알수없는칸"]))
print("모르는 파일 →", 알수없는것, "(사람에게 묻는다)")

시세   → ohlcv_stock     ohlcv_stock 0.98 · ohlcv_index 0.56 · financial 0.02
뉴스   → news            news 1.00 · financial 0.00 · macro 0.00
거시   → macro           macro 0.98 · news 0.02 · financial 0.00

모르는 파일 → None (사람에게 묻는다)


---

## 8. 세션마다 도는 것

```bash
python scripts/check_inbox.py            # 로컬 + HuggingFace 를 훑고 새 것만 들인다
python scripts/check_inbox.py --dry-run  # 검사만 하고 DB 에는 안 담는다
python scripts/check_inbox.py --force    # 규격을 고친 뒤 다시 검사한다
```

지금 HuggingFace `inbox/` 는 비어 있습니다 — 팀원이 아직 올린 것이 없습니다. 올라오면 이
스크립트가 찾아서 내려받고, 검사하고, 담고, 보고서를 냅니다.

---

## 남은 것

- **`financial` 규격은 실제 파일로 시험하지 못했습니다.** DART 재무제표를 아직 안 받았습니다
  (3차로 미뤘습니다). 단위 시험은 있지만 왕복 시험은 없습니다.
- **FRED 계열별 공표 일정**을 확인하지 못했습니다. 월 45일·분기 130일은 추정입니다.
- **같은 종목코드 가정이 프로젝트 다른 곳에도 있습니다** — `ingest/clients/dart_data.py` 의
  `STOCK_CODE_PATTERN`, `ingest/clients/yf_data.py` 가 아직 `^\\d{6}$` 입니다. 이번 PR 범위
  밖이라 두었습니다.
- **합격분을 `daily_price` 와 맞대 보는 일**(규격의 `target.compareWith`)은 아직 안 했습니다.
  적재까지가 이번 몫이고, 대조는 다음입니다.